# E-Commerce Sales & Profit Analytics
### End-to-End Data Analysis Portfolio Project
**Author:** Data Analytics Portfolio Project  
**Tools & Technologies:** Python, Pandas, NumPy, Matplotlib, Seaborn, SQLite, SQL

---

## Project Overview
This project analyzes the historical sales and profit performance of an e-commerce retail store (Superstore dataset). The goal is to uncover revenue drivers, identify profit leakages, evaluate category and regional performance, and derive actionable business strategies.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [ ]:
# Load cleaned dataset
data_path = os.path.join('..', 'data', 'clean_superstore.csv') if os.path.exists(os.path.join('..', 'data', 'clean_superstore.csv')) else os.path.join('data', 'clean_superstore.csv')
df = pd.read_csv(data_path)
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 2. High-Level Key Performance Indicators (KPIs)

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
profit_margin = (total_profit / total_sales) * 100
avg_order_value = total_sales / total_orders

print("=" * 45)
print("          EXECUTIVE KPI SUMMARY")
print("=" * 45)
print(f"Total Revenue:        ${total_sales:,.2f}")
print(f"Total Profit:         ${total_profit:,.2f}")
print(f"Profit Margin:        {profit_margin:.2f}%")
print(f"Total Unique Orders:  {total_orders:,}")
print(f"Total Unique Customers: {total_customers:,}")
print(f"Avg Order Value:      ${avg_order_value:,.2f}")
print("=" * 45)

## 3. Monthly Sales & Profit Trend Analysis

In [ ]:
monthly = df.set_index('Order Date').resample('M').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
monthly['Year_Month'] = monthly['Order Date'].dt.strftime('%Y-%m')

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.plot(monthly['Year_Month'], monthly['Sales'], color='#2563eb', marker='o', linewidth=2.5, label='Sales ($)')
ax2.plot(monthly['Year_Month'], monthly['Profit'], color='#16a34a', marker='s', linewidth=2.5, label='Profit ($)')

ax1.set_title('Monthly Sales & Profit Performance Over Time', fontsize=16, fontweight='bold', pad=15)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Sales ($)', color='#2563eb', fontsize=12)
ax2.set_ylabel('Profit ($)', color='#16a34a', fontsize=12)

ticks_idx = range(0, len(monthly), 3)
ax1.set_xticks(ticks_idx)
ax1.set_xticklabels([monthly['Year_Month'].iloc[i] for i in ticks_idx], rotation=45)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Category & Sub-Category Performance Analysis

In [ ]:
cat_perf = df.groupby('Category').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
cat_perf['Profit_Margin_%'] = (cat_perf['Profit'] / cat_perf['Sales']) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=cat_perf, x='Category', y='Sales', ax=ax1, palette=['#3b82f6', '#10b981', '#f59e0b'])
ax1.set_title('Sales by Category', fontsize=14, fontweight='bold')
ax1.set_ylabel('Total Sales ($)')

sns.barplot(data=cat_perf, x='Category', y='Profit', ax=ax2, palette=['#3b82f6', '#10b981', '#f59e0b'])
ax2.set_title('Profit by Category', fontsize=14, fontweight='bold')
ax2.set_ylabel('Total Profit ($)')

plt.tight_layout()
plt.show()
display(cat_perf)

## 5. Discount Impact & Profitability Analysis

In [ ]:
bins = [-0.01, 0, 0.10, 0.20, 0.30, 1.0]
labels = ['0% (None)', '1-10%', '11-20%', '21-30%', '30%+']
df['Discount_Tier'] = pd.cut(df['Discount'], bins=bins, labels=labels)

disc_summary = df.groupby('Discount_Tier').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Order_Count=('Order ID', 'count')
).reset_index()
disc_summary['Profit_Margin_%'] = (disc_summary['Total_Profit'] / disc_summary['Total_Sales']) * 100

plt.figure(figsize=(10, 5))
colors = ['#16a34a' if m > 0 else '#dc2626' for m in disc_summary['Profit_Margin_%']]
sns.barplot(data=disc_summary, x='Discount_Tier', y='Profit_Margin_%', palette=colors)
plt.title('Profit Margin % by Discount Tier (Critical Threshold Alert)', fontsize=14, fontweight='bold')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.ylabel('Profit Margin (%)')
plt.xlabel('Discount Tier')
plt.tight_layout()
plt.show()
display(disc_summary)

## 6. SQL Business Analysis (SQLite Integration)

In [ ]:
db_path = os.path.join('..', 'data', 'ecommerce.db') if os.path.exists(os.path.join('..', 'data', 'ecommerce.db')) else os.path.join('data', 'ecommerce.db')
conn = sqlite3.connect(db_path)

# Query: Top 10 Profitable Customers using Window Function
sql_query = """
WITH Customer_Metrics AS (
    SELECT 
        "Customer Name",
        Segment,
        COUNT(DISTINCT "Order ID") AS total_orders,
        ROUND(SUM(Sales), 2) AS total_revenue,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS profit_margin_pct,
        DENSE_RANK() OVER (ORDER BY SUM(Profit) DESC) as profit_rank
    FROM sales
    GROUP BY "Customer Name", Segment
)
SELECT * FROM Customer_Metrics
WHERE profit_rank <= 10;
"""

top_customers_sql = pd.read_sql_query(sql_query, conn)
conn.close()
top_customers_sql

## 7. Executive Insights & Business Recommendations

1. **Eliminate Aggressive Discounts (>20%):**
   - Discounts over 20% erode all profit margins, leading to an aggregate loss of **-$135,000+**.
   - Cap standard sales discounts at 15-20% max.

2. **Restructure Tables & Bookcases Sub-Categories:**
   - `Tables` is currently running a net loss (-$17.7k) despite $206k in revenue.
   - Re-evaluate supplier contracts and shipping surcharges for heavy items.

3. **Double Down on Technology & Office Supplies:**
   - Technology generates over **$145k in profit** (highest among all categories).
   - Expand high-margin catalog in Copiers, Accessories, and Paper.

4. **Target High-Value Customer Retention:**
   - The top 10% of customers drive a disproportionate share of operating profits. Introduce VIP loyalty programs and dedicated B2B account support.